# MCTNet training from Google Drive

Ce notebook charge les datasets `.npz/.json` prepares auparavant,
instancie MCTNet, puis lance l'entrainement et l'evaluation.


In [ ]:
import json
import sys
from pathlib import Path
from types import SimpleNamespace

try:
    from google.colab import drive
    drive.mount('/content/drive')
    print('Google Drive monte.')
except ImportError:
    print('Google Colab non detecte.')


In [ ]:
# Adapte ces chemins a ton Drive.
PROJECT_DIR = Path('/content/drive/MyDrive/New project/python')
DATA_DIR = Path('/content/drive/MyDrive/mctnet_crop_mapping_2021/processed')
OUTPUT_DIR = DATA_DIR / 'training_runs' / 'arkansas_run_01'

DATASET_NPZ = DATA_DIR / 'arkansas_mctnet_dataset.npz'
METADATA_JSON = DATA_DIR / 'arkansas_mctnet_dataset.json'

sys.path.insert(0, str(PROJECT_DIR))

print('PROJECT_DIR =', PROJECT_DIR)
print('DATASET_NPZ exists =', DATASET_NPZ.exists())
print('METADATA_JSON exists =', METADATA_JSON.exists())
print('OUTPUT_DIR =', OUTPUT_DIR)


In [ ]:
from train_mctnet import load_bundle, train_model, set_seed

args = SimpleNamespace(
    dataset_npz=str(DATASET_NPZ),
    metadata_json=str(METADATA_JSON),
    output_dir=str(OUTPUT_DIR),
    checkpoint_path=str(OUTPUT_DIR / 'best_mctnet.pt'),
    epochs=100,
    batch_size=32,
    learning_rate=1e-3,
    weight_decay=1e-4,
    dropout=0.1,
    n_stages=3,
    n_heads=5,
    kernel_size=3,
    seed=2021,
    num_workers=0,
    early_stopping_patience=15,
    cpu=False,
    no_alpe=False,
    no_mask=False,
    no_cnn=False,
    no_trans=False,
)

set_seed(args.seed)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

bundle = load_bundle(DATASET_NPZ)
metadata = json.loads(METADATA_JSON.read_text(encoding='utf-8'))

print('State:', metadata['state_name'])
print('Classes:', metadata['class_name_to_index'])


In [ ]:
model, metrics_payload = train_model(args=args, bundle=bundle, metadata=metadata)

metrics_path = OUTPUT_DIR / 'metrics.json'
metrics_path.write_text(json.dumps(metrics_payload, indent=2), encoding='utf-8')

print('Best validation metrics:')
print(json.dumps(metrics_payload['best_val'], indent=2))
print('Test metrics at best validation epoch:')
print(json.dumps(metrics_payload['test_at_best_val'], indent=2))
print('Checkpoint:', OUTPUT_DIR / 'best_mctnet.pt')
print('Metrics:', metrics_path)


In [ ]:
# Exemple d'ablation, si tu veux tester les variantes du papier.
# args.no_alpe = True
# args.no_mask = True
# args.no_cnn = True
# args.no_trans = True
